# 02 — Heston characteristic function + Carr-Madan FFT

Validate the Albrecher 'little Heston trap' characteristic function against brute force numerical integration and confirm the FFT pricer agrees to 4+ decimal places.

## Context

The Heston characteristic function of the log-spot is

$$\phi(u; T) = \exp\!\big[ C(u, T) + D(u, T)\, v_0 + iu \log S_0 + iu (r - q) T \big].$$

Two algebraically equivalent closed forms exist. The original Heston (1993) form has a branch-cut bug at long maturities; Albrecher–Mayer–Schoutens–Tistaert (2007) — the *Little Heston Trap* — swap the sign inside the auxiliary $g$ to keep it on the principal branch. Carr–Madan (1999) Fourier-inverts $\phi$ on a uniform log-strike grid to deliver the full smile in a single FFT.

This notebook validates both the characteristic function and the FFT pricer against brute-force numerical integration.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import quad

from volengine.models.heston import HestonParameters, heston_char_fn, heston_vanilla_price
from volengine.surfaces import implied_vol, black_scholes_price

In [ ]:
p = HestonParameters(kappa=1.5, theta=0.04, xi=0.5, rho=-0.7, v0=0.04)
S0, r, q, T = 100.0, 0.03, 0.01, 0.5

def heston_call_quad(K, alpha=1.5):
    def integrand(v):
        phi = heston_char_fn(v - 1j * (alpha + 1), T, S0, r, q, p)
        psi = np.exp(-r * T) * phi / (alpha**2 + alpha - v**2 + 1j * (2 * alpha + 1) * v)
        return (np.exp(-1j * v * np.log(K)) * psi).real
    val, _ = quad(integrand, 0, 200.0, limit=200)
    return np.exp(-alpha * np.log(K)) / np.pi * val

Ks = np.linspace(80, 120, 9)
fft_prices = heston_vanilla_price(Ks, T, S0, r, q, p)
quad_prices = [heston_call_quad(K) for K in Ks]
pd.DataFrame({'K': Ks, 'FFT': fft_prices, 'quad': quad_prices,
              'diff': np.array(fft_prices) - np.array(quad_prices)})

## FFT vs. quadrature

The table below shows the FFT and `scipy.integrate.quad` prices side by side. Differences should sit at $\le 10^{-4}$ across all strikes — that's the 'four decimal places' benchmark every Heston implementation chases.

In [ ]:
# Recover implied vol smile under Heston.
ivs = [implied_vol(float(P), S0, K, T, r, q, 'call') for P, K in zip(fft_prices, Ks)]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(Ks / S0, np.array(ivs) * 100, 'o-', color='C3', lw=2)
ax.axvline(1.0, color='grey', ls=':', lw=0.8, label='ATM (K = S0)')
ax.set_xlabel('moneyness  K / S0')
ax.set_ylabel('Heston implied vol (%)')
ax.set_title('Heston implied-vol smile at T = 0.5y\n'
             r'$\kappa$=1.5, $\theta$=0.04, $\xi$=0.5, $\rho$=$-$0.7, $v_0$=0.04')
ax.grid(True, alpha=0.3)
ax.legend()
fig.text(0.5, -0.03,
         'Downward slope (negative skew) is driven by the spot/vol correlation '
         r'$\rho < 0$; the overall level by $v_0$. The curvature comes from the '
         r'vol-of-vol $\xi$. Heston smiles flatten too fast with maturity — the '
         'empirical gap that motivates rough volatility (notebook 05).',
         ha='center', fontsize=9, wrap=True)
fig.tight_layout()
fig.savefig('../results/figures/heston_smile.png', dpi=120, bbox_inches='tight')
plt.show()

## Heston smile

**Figure.** Implied-vol smile under SPX-typical Heston parameters ($\kappa{=}1.5,\ \theta{=}0.04,\ \xi{=}0.5,\ \rho{=}-0.7,\ v_0{=}0.04$), $T = 0.5\,$y. The negative skew is set by $\rho < 0$; the level by $v_0$. Heston produces smiles that are too flat at long maturities — the empirical motivation for the rough-vol extension in notebook 05.